<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/03_Statistical_%26_Data_Characterization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
# ==============================================================================
# NOTEBOOK 03 — STATISTICAL & DATA CHARACTERIZATION
# ==============================================================================
#
# PURPOSE
# -------
# This notebook constructs the statistical reference layer required by the
# SPP-GAN framework.
#
# The notebook characterizes:
#   1. Dataset-level properties
#   2. Feature types
#   3. Numerical distributions
#   4. Categorical distributions
#   5. Missingness
#   6. Cardinality and entropy
#   7. Pearson dependency
#   8. Spearman dependency
#   9. Categorical dependency
#  10. Feature-level statistical profiles
#  11. Dataset-level statistical profiles
#  12. SPP-GAN statistical reference
#  13. Machine-readable statistical guidance
#
# IMPORTANT METHODOLOGICAL POLICY
# --------------------------------
# Statistical reference information is derived from the TRAINING split only.
#
# Validation and test data are NOT used to construct:
#   - distributions
#   - frequencies
#   - entropy
#   - correlations
#   - dependency matrices
#   - SPP-GAN statistical guidance
#
# Notebook 02 remains frozen.
#
# ==============================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import math
import os
import warnings
import hashlib

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("=" * 100)
print("NOTEBOOK 03 — STATISTICAL & DATA CHARACTERIZATION")
print("=" * 100)

print("\nScope:")
print("  • Training-only statistical characterization")
print("  • Publication-grade statistical reference construction")
print("  • RAM-safe Colab implementation")
print("  • No model training")
print("  • No synthetic data generation")
print("  • No validation/test leakage")

NOTEBOOK 03 — STATISTICAL & DATA CHARACTERIZATION

Scope:
  • Training-only statistical characterization
  • Publication-grade statistical reference construction
  • RAM-safe Colab implementation
  • No model training
  • No synthetic data generation
  • No validation/test leakage


In [28]:
# ==============================================================================
# SECTION 2 — LOAD PROCESSED DATA
# ==============================================================================
#
# PURPOSE
# -------
# Load the canonical TRAINING datasets produced and validated by Notebook 02.
#
# METHODOLOGICAL POLICY
# ---------------------
# 1. Only TRAIN data are loaded for statistical characterization.
# 2. Validation and test data are NOT used in Notebook 03 statistical
#    reference construction.
# 3. Targets remain part of the modeling dataset and are treated according
#    to the established target policy.
# 4. Identifiers are retained only where present for audit purposes and are
#    excluded from statistical feature analysis later.
# 5. Provenance is audit-only and is never treated as a statistical feature.
# 6. No synthetic data are generated in this section.
# 7. No model is trained in this section.
#
# ==============================================================================

from pathlib import Path
import pandas as pd
import numpy as np


print("=" * 100)
print("SECTION 2 — LOAD PROCESSED DATA")
print("=" * 100)


# ==============================================================================
# 2.1 GOOGLE DRIVE MOUNT
# ==============================================================================

print("\n[2.1] Checking Google Drive mount")
print("-" * 100)

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        "Google Colab Drive interface could not be imported. "
        "This notebook is designed for Google Colab."
    ) from exc


DRIVE_ROOT = Path("/content/drive")
MYDRIVE_ROOT = DRIVE_ROOT / "MyDrive"


if not MYDRIVE_ROOT.exists():

    print("Google Drive is not mounted.")
    print("Mounting Google Drive...")

    drive.mount("/content/drive")


if not MYDRIVE_ROOT.exists():

    raise FileNotFoundError(
        "Google Drive mount failed.\n"
        f"Expected directory:\n{MYDRIVE_ROOT}"
    )


print("✓ Google Drive is mounted.")
print(f"✓ MyDrive : {MYDRIVE_ROOT}")


# ==============================================================================
# 2.2 PROJECT PATH CONFIGURATION
# ==============================================================================

print("\n[2.2] Configuring project paths")
print("-" * 100)


PROJECT_ROOT = (
    MYDRIVE_ROOT
    / "SPP_GAN_Research"
)


NB02_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_02"
)


NB03_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_03"
)


NB03_RESULTS_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebook_03"
)


print(f"Project root : {PROJECT_ROOT}")
print(f"Notebook 02  : {NB02_ROOT}")
print(f"Notebook 03  : {NB03_ROOT}")


# ==============================================================================
# 2.3 PROJECT ROOT VALIDATION
# ==============================================================================

print("\n[2.3] Validating project root")
print("-" * 100)


if not PROJECT_ROOT.is_dir():

    raise FileNotFoundError(
        "SPP-GAN project root does not exist.\n"
        f"Expected:\n{PROJECT_ROOT}"
    )


print("✓ SPP-GAN project root found.")


# ==============================================================================
# 2.4 NOTEBOOK 02 ROOT VALIDATION
# ==============================================================================

print("\n[2.4] Validating Notebook 02 artifacts")
print("-" * 100)


if not NB02_ROOT.is_dir():

    raise FileNotFoundError(
        "Notebook 02 processed-data directory was not found.\n"
        f"Expected:\n{NB02_ROOT}\n\n"
        "Notebook 02 Section 25 reports that this directory should "
        "exist. Verify that Google Drive has finished synchronizing."
    )


print("✓ Notebook 02 processed-data root found.")


# ==============================================================================
# 2.5 CANONICAL NOTEBOOK 02 DIRECTORIES
# ==============================================================================

NB02_NATIVE_ROOT = (
    NB02_ROOT
    / "native"
)

NB02_ENCODED_ROOT = (
    NB02_ROOT
    / "encoded"
)

NB02_SPLITS_ROOT = (
    NB02_ROOT
    / "splits"
)

NB02_PREPROCESSORS_ROOT = (
    NB02_ROOT
    / "preprocessors"
)

NB02_SCHEMAS_ROOT = (
    NB02_ROOT
    / "schemas"
)

NB02_FEATURE_MAPPING_ROOT = (
    NB02_ROOT
    / "feature_mapping"
)


REQUIRED_NB02_DIRECTORIES = {
    "native": NB02_NATIVE_ROOT,
    "encoded": NB02_ENCODED_ROOT,
    "splits": NB02_SPLITS_ROOT,
    "preprocessors": NB02_PREPROCESSORS_ROOT,
    "schemas": NB02_SCHEMAS_ROOT,
    "feature_mapping": NB02_FEATURE_MAPPING_ROOT,
}


print("\nCanonical Notebook 02 directories:")

for name, path in REQUIRED_NB02_DIRECTORIES.items():

    if not path.is_dir():

        raise FileNotFoundError(
            f"Required Notebook 02 directory missing:\n{path}"
        )

    print(f"  ✓ {name:<18}: {path}")


print("\n✓ Notebook 02 directory structure validated.")


# ==============================================================================
# 2.6 NOTEBOOK 03 OUTPUT DIRECTORIES
# ==============================================================================

print("\n[2.5] Preparing Notebook 03 output directories")
print("-" * 100)


NB03_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

NB03_RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


for subdir in [
    "profiles",
    "statistics",
    "correlations",
    "dependencies",
    "reference",
    "guidance",
    "reports",
    "schemas",
]:

    (NB03_ROOT / subdir).mkdir(
        parents=True,
        exist_ok=True
    )


print("✓ Notebook 03 output directories ready.")


# ==============================================================================
# 2.7 DATASET REGISTRY
# ==============================================================================

print("\n[2.6] Initializing dataset registry")
print("-" * 100)


DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]


TARGET_COLUMNS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}


IDENTIFIER_COLUMNS = {
    "adult_income": [],
    "bank_marketing": [],
    "diabetes_130us": [
        "encounter_id",
        "patient_nbr",
    ],
}


PROVENANCE_COLUMN = "__original_row_id__"


print(f"Datasets : {DATASET_IDS}")


# ==============================================================================
# 2.8 NATIVE DATASET SPLIT DISCOVERY
# ==============================================================================

print("\n[2.7] Discovering canonical TRAIN datasets")
print("-" * 100)


def discover_dataset_split_file(
    dataset_id,
    split_name="train"
):
    """
    Locate a canonical Notebook 02 native dataset split.

    The function first checks the expected dataset directory and then
    performs a controlled recursive search within that dataset directory.
    """

    dataset_dir = (
        NB02_NATIVE_ROOT
        / dataset_id
    )


    if not dataset_dir.is_dir():

        raise FileNotFoundError(
            f"Notebook 02 native dataset directory not found:\n"
            f"{dataset_dir}"
        )


    # --------------------------------------------------------------------------
    # Expected serialization formats
    # --------------------------------------------------------------------------

    extensions = [
        ".parquet",
        ".csv",
        ".pkl",
        ".pickle",
        ".feather",
    ]


    # --------------------------------------------------------------------------
    # Direct expected filenames
    # --------------------------------------------------------------------------

    for extension in extensions:

        candidate = (
            dataset_dir
            / f"{split_name}{extension}"
        )

        if candidate.is_file():
            return candidate


    # --------------------------------------------------------------------------
    # Controlled recursive fallback
    # --------------------------------------------------------------------------

    matches = []

    for extension in extensions:

        matches.extend(
            dataset_dir.rglob(
                f"{split_name}{extension}"
            )
        )


    if matches:

        matches = sorted(
            set(matches),
            key=lambda p: str(p).lower()
        )

        return matches[0]


    # --------------------------------------------------------------------------
    # Diagnostic information
    # --------------------------------------------------------------------------

    available_files = sorted(
        [
            p
            for p in dataset_dir.rglob("*")
            if p.is_file()
        ],
        key=lambda p: str(p).lower()
    )


    diagnostic = "\n".join(
        f"  - {p}"
        for p in available_files[:50]
    )


    raise FileNotFoundError(
        f"No '{split_name}' split found for dataset "
        f"'{dataset_id}'.\n\n"
        f"Dataset directory:\n{dataset_dir}\n\n"
        f"Available files:\n"
        f"{diagnostic if diagnostic else '  <none>'}"
    )


# ==============================================================================
# 2.9 DATAFRAME LOADER
# ==============================================================================

def load_dataframe(path):
    """
    Load a persisted Notebook 02 dataframe.
    """

    if not path.is_file():

        raise FileNotFoundError(
            f"Data file does not exist:\n{path}"
        )


    suffix = path.suffix.lower()


    if suffix == ".parquet":

        return pd.read_parquet(path)


    elif suffix == ".csv":

        return pd.read_csv(path)


    elif suffix in [".pkl", ".pickle"]:

        return pd.read_pickle(path)


    elif suffix == ".feather":

        return pd.read_feather(path)


    else:

        raise ValueError(
            f"Unsupported dataframe format:\n{path}"
        )


# ==============================================================================
# 2.10 LOAD TRAINING DATA ONLY
# ==============================================================================

print("\n[2.8] Loading TRAINING data only")
print("-" * 100)


TRAIN_STATISTICAL_DATASETS = {}
TRAIN_DATASET_PATHS = {}


for dataset_id in DATASET_IDS:

    print(f"\nLoading: {dataset_id}")


    # --------------------------------------------------------------------------
    # Locate train file
    # --------------------------------------------------------------------------

    train_path = discover_dataset_split_file(
        dataset_id=dataset_id,
        split_name="train"
    )


    print(f"  File : {train_path}")


    # --------------------------------------------------------------------------
    # Load dataframe
    # --------------------------------------------------------------------------

    df = load_dataframe(
        train_path
    )


    # --------------------------------------------------------------------------
    # Basic dataframe validation
    # --------------------------------------------------------------------------

    if not isinstance(
        df,
        pd.DataFrame
    ):

        raise TypeError(
            f"Loaded object for '{dataset_id}' "
            "is not a pandas DataFrame."
        )


    if df.empty:

        raise ValueError(
            f"Training dataframe for '{dataset_id}' is empty."
        )


    # --------------------------------------------------------------------------
    # Store canonical objects
    # --------------------------------------------------------------------------

    TRAIN_STATISTICAL_DATASETS[
        dataset_id
    ] = df


    TRAIN_DATASET_PATHS[
        dataset_id
    ] = str(train_path)


    print(
        f"  Rows    : {df.shape[0]:,}"
    )

    print(
        f"  Columns : {df.shape[1]:,}"
    )


# ==============================================================================
# 2.11 DATASET COMPLETENESS VALIDATION
# ==============================================================================

print("\n[2.9] Validating loaded datasets")
print("-" * 100)


if set(
    TRAIN_STATISTICAL_DATASETS.keys()
) != set(DATASET_IDS):

    raise RuntimeError(
        "TRAIN_STATISTICAL_DATASETS is incomplete.\n"
        f"Expected : {DATASET_IDS}\n"
        f"Loaded   : "
        f"{list(TRAIN_STATISTICAL_DATASETS.keys())}"
    )


if set(
    TRAIN_DATASET_PATHS.keys()
) != set(DATASET_IDS):

    raise RuntimeError(
        "TRAIN_DATASET_PATHS is incomplete."
    )


print("✓ All required training datasets loaded.")


# ==============================================================================
# 2.12 TARGET VALIDATION
# ==============================================================================

print("\n[2.10] Validating target columns")
print("-" * 100)


for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[
        dataset_id
    ]

    target_column = TARGET_COLUMNS[
        dataset_id
    ]


    if target_column not in df.columns:

        raise RuntimeError(
            f"Target column '{target_column}' "
            f"is missing from dataset '{dataset_id}'."
        )


    print(
        f"  ✓ {dataset_id:<20} "
        f"Target = {target_column}"
    )


# ==============================================================================
# 2.13 IDENTIFIER VALIDATION
# ==============================================================================

print("\n[2.11] Validating identifier policy")
print("-" * 100)


for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[
        dataset_id
    ]

    expected_identifiers = IDENTIFIER_COLUMNS[
        dataset_id
    ]


    missing_identifiers = [
        column
        for column in expected_identifiers
        if column not in df.columns
    ]


    if missing_identifiers:

        raise RuntimeError(
            f"Expected identifier columns missing from "
            f"'{dataset_id}': {missing_identifiers}"
        )


    if expected_identifiers:

        print(
            f"  ✓ {dataset_id:<20} "
            f"Identifiers = {expected_identifiers}"
        )

    else:

        print(
            f"  ✓ {dataset_id:<20} "
            f"Identifiers = None"
        )


# ==============================================================================
# 2.14 PROVENANCE POLICY
# ==============================================================================

print("\n[2.12] Validating provenance policy")
print("-" * 100)


for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[
        dataset_id
    ]


    if PROVENANCE_COLUMN in df.columns:

        print(
            f"  ✓ {dataset_id:<20} "
            f"Provenance detected: {PROVENANCE_COLUMN} "
            f"(audit-only)"
        )

    else:

        print(
            f"  ✓ {dataset_id:<20} "
            f"No provenance column present"
        )


# ==============================================================================
# 2.15 FINAL SECTION 2 SUMMARY
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 2 — LOAD PROCESSED DATA SUMMARY")
print("=" * 100)


print(
    f"\nNotebook 02 root : {NB02_ROOT}"
)

print(
    f"Native root      : {NB02_NATIVE_ROOT}"
)


print("\nLoaded TRAIN datasets:")


for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[
        dataset_id
    ]

    print(
        f"  {dataset_id:<20} | "
        f"Rows = {len(df):>8,} | "
        f"Columns = {len(df.columns):>4} | "
        f"Target = {TARGET_COLUMNS[dataset_id]}"
    )


# ==============================================================================
# FINAL GATE
# ==============================================================================

print("\n" + "-" * 100)
print("SECTION 2 FINAL VALIDATION")
print("-" * 100)

print("✓ Google Drive mounted")
print("✓ SPP-GAN project root verified")
print("✓ Notebook 02 root verified")
print("✓ Notebook 02 native directory verified")
print("✓ All three TRAIN datasets loaded")
print("✓ Target columns verified")
print("✓ Identifier policy verified")
print("✓ Provenance policy verified")
print("✓ TRAIN-ONLY statistical policy preserved")
print("✓ No validation/test data used")
print("✓ No synthetic data generated")
print("✓ No model training performed")

print("\nSECTION 2 STATUS: PASS")

SECTION 2 — LOAD PROCESSED DATA

[2.1] Checking Google Drive mount
----------------------------------------------------------------------------------------------------
✓ Google Drive is mounted.
✓ MyDrive : /content/drive/MyDrive

[2.2] Configuring project paths
----------------------------------------------------------------------------------------------------
Project root : /content/drive/MyDrive/SPP_GAN_Research
Notebook 02  : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
Notebook 03  : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_03

[2.3] Validating project root
----------------------------------------------------------------------------------------------------
✓ SPP-GAN project root found.

[2.4] Validating Notebook 02 artifacts
----------------------------------------------------------------------------------------------------


FileNotFoundError: Notebook 02 processed-data directory was not found.
Expected:
/content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02

Notebook 02 Section 25 reports that this directory should exist. Verify that Google Drive has finished synchronizing.

In [24]:
# ==============================================================================
# NOTEBOOK 02 ARTIFACT LOCATION DIAGNOSTIC
# ==============================================================================

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Research")

print("=" * 100)
print("NOTEBOOK 02 ARTIFACT LOCATION DIAGNOSTIC")
print("=" * 100)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"SPP-GAN project root not found:\n{PROJECT_ROOT}"
    )

print(f"\nProject root found:")
print(PROJECT_ROOT)

print("\n" + "-" * 100)
print("TOP-LEVEL PROJECT STRUCTURE")
print("-" * 100)

for item in sorted(PROJECT_ROOT.iterdir(), key=lambda x: x.name.lower()):
    kind = "DIR " if item.is_dir() else "FILE"
    print(f"{kind} | {item.name}")


print("\n" + "-" * 100)
print("SEARCHING FOR NOTEBOOK 02 DIRECTORIES")
print("-" * 100)

nb02_dirs = [
    p for p in PROJECT_ROOT.rglob("*")
    if p.is_dir() and "notebook_02" in p.name.lower()
]

if nb02_dirs:
    for p in sorted(nb02_dirs, key=lambda x: str(x).lower()):
        print(f"FOUND: {p}")
else:
    print("No directory containing 'notebook_02' was found.")


print("\n" + "-" * 100)
print("SEARCHING FOR NOTEBOOK 02 KNOWN ARTIFACTS")
print("-" * 100)

known_artifacts = [
    "split_manifest.csv",
    "split_summary.csv",
    "split_integrity.csv",
    "target_validation.csv",
    "transformation_report.csv",
    "preprocessing_schema.json",
    "train_fitted_preprocessor.joblib",
    "notebook_02_completion_metadata.json",
]

found = {}

for artifact in known_artifacts:

    matches = list(PROJECT_ROOT.rglob(artifact))

    if matches:
        found[artifact] = matches

        for path in matches:
            print(f"FOUND | {artifact}")
            print(f"       {path}")
    else:
        print(f"NOT FOUND | {artifact}")


print("\n" + "-" * 100)
print("SEARCHING FOR NATIVE TRAIN SPLITS")
print("-" * 100)

dataset_ids = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

for dataset_id in dataset_ids:

    print(f"\n{dataset_id}:")

    matches = []

    for pattern in [
        "train.parquet",
        "train.csv",
        "train.pkl",
        "train.pickle",
        "train.feather",
    ]:
        matches.extend(
            PROJECT_ROOT.rglob(pattern)
        )

    dataset_matches = [
        p for p in matches
        if dataset_id.lower() in str(p).lower()
    ]

    if dataset_matches:
        for path in sorted(
            set(dataset_matches),
            key=lambda x: str(x).lower()
        ):
            print(f"  FOUND: {path}")
    else:
        print("  No train split found.")


print("\n" + "=" * 100)
print("DIAGNOSTIC COMPLETE")
print("=" * 100)

NOTEBOOK 02 ARTIFACT LOCATION DIAGNOSTIC

Project root found:
/content/drive/MyDrive/SPP_GAN_Research

----------------------------------------------------------------------------------------------------
TOP-LEVEL PROJECT STRUCTURE
----------------------------------------------------------------------------------------------------
DIR  | data
DIR  | results

----------------------------------------------------------------------------------------------------
SEARCHING FOR NOTEBOOK 02 DIRECTORIES
----------------------------------------------------------------------------------------------------
No directory containing 'notebook_02' was found.

----------------------------------------------------------------------------------------------------
SEARCHING FOR NOTEBOOK 02 KNOWN ARTIFACTS
----------------------------------------------------------------------------------------------------
NOT FOUND | split_manifest.csv
NOT FOUND | split_summary.csv
NOT FOUND | split_integrity.csv
NOT FOUND | 

In [3]:
# ==============================================================================
# SECTION 3 — VALIDATE INPUT SCHEMAS
# ==============================================================================

print("=" * 100)
print("SECTION 3 — VALIDATE INPUT SCHEMAS")
print("=" * 100)

INPUT_SCHEMA_REPORTS = []
INPUT_SCHEMA_SUMMARY = []

for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    target = TARGET_COLUMNS[dataset_id]
    identifiers = IDENTIFIER_COLUMNS[dataset_id]

    failures = []

    if target not in df.columns:
        failures.append("target_missing")

    if PROVENANCE_COLUMN not in df.columns:
        failures.append("provenance_missing")

    for identifier in identifiers:
        if identifier in df.columns:
            failures.append(
                f"identifier_present_in_native_data:{identifier}"
            )

    modeling_columns = [
        c for c in df.columns
        if c != PROVENANCE_COLUMN
        and c not in identifiers
    ]

    if target not in modeling_columns:
        failures.append("target_not_in_modeling_columns")

    duplicate_columns = df.columns[df.columns.duplicated()].tolist()

    if duplicate_columns:
        failures.append("duplicate_columns")

    finite_numeric_failure = False

    numeric_columns = df[modeling_columns].select_dtypes(
        include=[np.number]
    ).columns.tolist()

    for col in numeric_columns:
        values = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        if np.isinf(values.to_numpy(dtype=float)).any():
            finite_numeric_failure = True
            break

    if finite_numeric_failure:
        failures.append("infinite_numeric_values")

    status = "PASS" if not failures else "FAIL"

    INPUT_SCHEMA_SUMMARY.append({
        "dataset_id": dataset_id,
        "rows": len(df),
        "total_columns": len(df.columns),
        "modeling_columns": len(modeling_columns),
        "numeric_columns": len(numeric_columns),
        "target_column": target,
        "identifier_columns_excluded": ", ".join(identifiers),
        "provenance_present": PROVENANCE_COLUMN in df.columns,
        "status": status,
        "failures": "; ".join(failures),
    })

    INPUT_SCHEMA_REPORTS.append({
        "dataset_id": dataset_id,
        "status": status,
        "failures": failures,
    })

    print(
        f"{dataset_id:20s} : {status}"
    )

INPUT_SCHEMA_SUMMARY_DF = pd.DataFrame(
    INPUT_SCHEMA_SUMMARY
)

if (INPUT_SCHEMA_SUMMARY_DF["status"] != "PASS").any():
    raise RuntimeError(
        "Notebook 02 input schema validation failed."
    )

print("\nSECTION 3 STATUS: PASS")

SECTION 3 — VALIDATE INPUT SCHEMAS


NameError: name 'TRAIN_STATISTICAL_DATASETS' is not defined

In [4]:
# ==============================================================================
# SECTION 4 — DATASET-LEVEL CHARACTERIZATION
# ==============================================================================

print("=" * 100)
print("SECTION 4 — DATASET-LEVEL CHARACTERIZATION")
print("=" * 100)

DATASET_CHARACTERIZATION_RECORDS = []

for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    target = TARGET_COLUMNS[dataset_id]
    identifiers = IDENTIFIER_COLUMNS[dataset_id]

    modeling_columns = [
        c for c in df.columns
        if c != PROVENANCE_COLUMN
        and c not in identifiers
    ]

    numeric_columns = df[modeling_columns].select_dtypes(
        include=[np.number]
    ).columns.tolist()

    categorical_columns = [
        c for c in modeling_columns
        if c not in numeric_columns
    ]

    target_non_null = df[target].notna().sum()
    target_unique = df[target].nunique(dropna=True)

    DATASET_CHARACTERIZATION_RECORDS.append({
        "dataset_id": dataset_id,
        "training_rows": len(df),
        "native_columns": len(df.columns),
        "modeling_columns": len(modeling_columns),
        "numeric_columns": len(numeric_columns),
        "categorical_columns": len(categorical_columns),
        "identifier_columns_excluded": len(identifiers),
        "target_column": target,
        "target_non_null": int(target_non_null),
        "target_missing": int(len(df) - target_non_null),
        "target_cardinality": int(target_unique),
        "memory_mb": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2),
            3
        ),
    })

DATASET_CHARACTERIZATION_DF = pd.DataFrame(
    DATASET_CHARACTERIZATION_RECORDS
)

print(
    DATASET_CHARACTERIZATION_DF.to_string(
        index=False
    )
)

print("\nSECTION 4 STATUS: PASS")

SECTION 4 — DATASET-LEVEL CHARACTERIZATION


NameError: name 'TRAIN_STATISTICAL_DATASETS' is not defined

In [5]:
# ==============================================================================
# SECTION 4 — DATASET-LEVEL CHARACTERIZATION
# ==============================================================================

print("=" * 100)
print("SECTION 4 — DATASET-LEVEL CHARACTERIZATION")
print("=" * 100)

DATASET_CHARACTERIZATION_RECORDS = []

for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    target = TARGET_COLUMNS[dataset_id]
    identifiers = IDENTIFIER_COLUMNS[dataset_id]

    modeling_columns = [
        c for c in df.columns
        if c != PROVENANCE_COLUMN
        and c not in identifiers
    ]

    numeric_columns = df[modeling_columns].select_dtypes(
        include=[np.number]
    ).columns.tolist()

    categorical_columns = [
        c for c in modeling_columns
        if c not in numeric_columns
    ]

    target_non_null = df[target].notna().sum()
    target_unique = df[target].nunique(dropna=True)

    DATASET_CHARACTERIZATION_RECORDS.append({
        "dataset_id": dataset_id,
        "training_rows": len(df),
        "native_columns": len(df.columns),
        "modeling_columns": len(modeling_columns),
        "numeric_columns": len(numeric_columns),
        "categorical_columns": len(categorical_columns),
        "identifier_columns_excluded": len(identifiers),
        "target_column": target,
        "target_non_null": int(target_non_null),
        "target_missing": int(len(df) - target_non_null),
        "target_cardinality": int(target_unique),
        "memory_mb": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2),
            3
        ),
    })

DATASET_CHARACTERIZATION_DF = pd.DataFrame(
    DATASET_CHARACTERIZATION_RECORDS
)

print(
    DATASET_CHARACTERIZATION_DF.to_string(
        index=False
    )
)

print("\nSECTION 4 STATUS: PASS")

SECTION 4 — DATASET-LEVEL CHARACTERIZATION


NameError: name 'TRAIN_STATISTICAL_DATASETS' is not defined

In [6]:
# ==============================================================================
# SECTION 6 — NUMERICAL DESCRIPTIVE STATISTICS
# ==============================================================================

print("=" * 100)
print("SECTION 6 — NUMERICAL DESCRIPTIVE STATISTICS")
print("=" * 100)

NUMERICAL_STATISTICS_RECORDS = []

for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    numeric_columns = FEATURE_TYPE_DF[
        (FEATURE_TYPE_DF["dataset_id"] == dataset_id) &
        (FEATURE_TYPE_DF["semantic_type"] == "numeric")
    ]["feature"].tolist()

    for feature in numeric_columns:

        s = pd.to_numeric(
            df[feature],
            errors="coerce"
        )

        non_missing = s.dropna()

        if len(non_missing) == 0:
            continue

        q = non_missing.quantile(
            [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
        )

        NUMERICAL_STATISTICS_RECORDS.append({
            "dataset_id": dataset_id,
            "feature": feature,
            "count": int(s.count()),
            "missing": int(s.isna().sum()),
            "missing_rate": float(s.isna().mean()),
            "mean": float(non_missing.mean()),
            "std": float(non_missing.std(ddof=1)),
            "min": float(non_missing.min()),
            "q01": float(q.loc[0.01]),
            "q05": float(q.loc[0.05]),
            "q25": float(q.loc[0.25]),
            "median": float(q.loc[0.50]),
            "q75": float(q.loc[0.75]),
            "q95": float(q.loc[0.95]),
            "q99": float(q.loc[0.99]),
            "max": float(non_missing.max()),
            "skewness": float(non_missing.skew()),
            "kurtosis": float(non_missing.kurtosis()),
            "n_unique": int(non_missing.nunique()),
        })

NUMERICAL_STATISTICS_DF = pd.DataFrame(
    NUMERICAL_STATISTICS_RECORDS
)

print(
    f"Numerical feature statistics generated: "
    f"{len(NUMERICAL_STATISTICS_DF)} rows"
)

print("\nSECTION 6 STATUS: PASS")

SECTION 6 — NUMERICAL DESCRIPTIVE STATISTICS


NameError: name 'TRAIN_STATISTICAL_DATASETS' is not defined

In [7]:
# ==============================================================================
# SECTION 7 — CATEGORICAL DESCRIPTIVE STATISTICS
# ==============================================================================

print("=" * 100)
print("SECTION 7 — CATEGORICAL DESCRIPTIVE STATISTICS")
print("=" * 100)

CATEGORICAL_STATISTICS_RECORDS = []

for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    categorical_columns = FEATURE_TYPE_DF[
        (FEATURE_TYPE_DF["dataset_id"] == dataset_id) &
        (FEATURE_TYPE_DF["semantic_type"] == "categorical")
    ]["feature"].tolist()

    for feature in categorical_columns:

        s = df[feature]

        value_counts = (
            s.astype("object")
            .value_counts(
                dropna=False,
                normalize=False
            )
        )

        total = len(s)

        missing_count = int(s.isna().sum())

        non_missing = s.dropna()

        if len(non_missing) > 0:
            top_value = non_missing.value_counts().index[0]
            top_count = int(
                non_missing.value_counts().iloc[0]
            )
        else:
            top_value = None
            top_count = 0

        CATEGORICAL_STATISTICS_RECORDS.append({
            "dataset_id": dataset_id,
            "feature": feature,
            "count": int(total),
            "missing": missing_count,
            "missing_rate": float(
                missing_count / total
            ),
            "n_unique": int(
                non_missing.nunique()
            ),
            "top_category": (
                str(top_value)
                if top_value is not None
                else None
            ),
            "top_count": top_count,
            "top_frequency": float(
                top_count / len(non_missing)
            ) if len(non_missing) > 0 else np.nan,
        })

CATEGORICAL_STATISTICS_DF = pd.DataFrame(
    CATEGORICAL_STATISTICS_RECORDS
)

print(
    f"Categorical feature statistics generated: "
    f"{len(CATEGORICAL_STATISTICS_DF)} rows"
)

print("\nSECTION 7 STATUS: PASS")

SECTION 7 — CATEGORICAL DESCRIPTIVE STATISTICS


NameError: name 'TRAIN_STATISTICAL_DATASETS' is not defined

In [8]:
# ==============================================================================
# SECTION 8 — MISSINGNESS CHARACTERIZATION
# ==============================================================================

print("=" * 100)
print("SECTION 8 — MISSINGNESS CHARACTERIZATION")
print("=" * 100)

MISSINGNESS_RECORDS = []

for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    target = TARGET_COLUMNS[dataset_id]
    identifiers = IDENTIFIER_COLUMNS[dataset_id]

    modeling_columns = [
        c for c in df.columns
        if c != PROVENANCE_COLUMN
        and c not in identifiers
    ]

    for feature in modeling_columns:

        missing_count = int(df[feature].isna().sum())
        total = len(df)

        MISSINGNESS_RECORDS.append({
            "dataset_id": dataset_id,
            "feature": feature,
            "missing_count": missing_count,
            "observed_count": int(total - missing_count),
            "missing_rate": float(
                missing_count / total
            ),
            "is_target": bool(feature == target),
        })

MISSINGNESS_DF = pd.DataFrame(
    MISSINGNESS_RECORDS
)

MISSINGNESS_SUMMARY_DF = (
    MISSINGNESS_DF
    .groupby("dataset_id", as_index=False)
    .agg(
        features=("feature", "count"),
        features_with_missing=(
            "missing_rate",
            lambda x: int((x > 0).sum())
        ),
        maximum_feature_missing_rate=(
            "missing_rate",
            "max"
        ),
        mean_feature_missing_rate=(
            "missing_rate",
            "mean"
        ),
    )
)

print(
    MISSINGNESS_SUMMARY_DF.to_string(
        index=False
    )
)

print("\nSECTION 8 STATUS: PASS")

SECTION 8 — MISSINGNESS CHARACTERIZATION


NameError: name 'TRAIN_STATISTICAL_DATASETS' is not defined

In [9]:
# ==============================================================================
# SECTION 9 — CARDINALITY / ENTROPY ANALYSIS
# ==============================================================================

print("=" * 100)
print("SECTION 9 — CARDINALITY / ENTROPY ANALYSIS")
print("=" * 100)

CARDINALITY_ENTROPY_RECORDS = []


def normalized_entropy(series):
    """
    Shannon entropy normalized to [0,1].
    """

    s = series.dropna()

    if len(s) == 0:
        return np.nan

    probabilities = (
        s.value_counts(normalize=True)
        .to_numpy(dtype=float)
    )

    entropy = -np.sum(
        probabilities *
        np.log2(
            np.clip(probabilities, 1e-15, None)
        )
    )

    cardinality = len(probabilities)

    if cardinality <= 1:
        return 0.0

    max_entropy = math.log2(cardinality)

    if max_entropy == 0:
        return 0.0

    return float(entropy / max_entropy)


for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    modeling_columns = [
        c for c in df.columns
        if c != PROVENANCE_COLUMN
        and c not in IDENTIFIER_COLUMNS[dataset_id]
    ]

    for feature in modeling_columns:

        s = df[feature]

        cardinality = int(
            s.dropna().nunique()
        )

        entropy = normalized_entropy(s)

        CARDINALITY_ENTROPY_RECORDS.append({
            "dataset_id": dataset_id,
            "feature": feature,
            "cardinality": cardinality,
            "normalized_entropy": entropy,
            "is_numeric": bool(
                pd.api.types.is_numeric_dtype(s)
            ),
            "is_target": bool(
                feature == TARGET_COLUMNS[dataset_id]
            ),
        })

CARDINALITY_ENTROPY_DF = pd.DataFrame(
    CARDINALITY_ENTROPY_RECORDS
)

print(
    f"Cardinality/entropy records: "
    f"{len(CARDINALITY_ENTROPY_DF)}"
)

print("\nSECTION 9 STATUS: PASS")

SECTION 9 — CARDINALITY / ENTROPY ANALYSIS


NameError: name 'TRAIN_STATISTICAL_DATASETS' is not defined

In [10]:
# ==============================================================================
# SECTION 10 — DISTRIBUTION CHARACTERIZATION
# ==============================================================================

print("=" * 100)
print("SECTION 10 — DISTRIBUTION CHARACTERIZATION")
print("=" * 100)

DISTRIBUTION_RECORDS = []

for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    for feature in [
        c for c in df.columns
        if c != PROVENANCE_COLUMN
        and c not in IDENTIFIER_COLUMNS[dataset_id]
    ]:

        s = df[feature]

        missing_rate = float(s.isna().mean())
        non_missing = s.dropna()

        record = {
            "dataset_id": dataset_id,
            "feature": feature,
            "missing_rate": missing_rate,
            "n_unique": int(non_missing.nunique()),
            "distribution_type": (
                "numeric"
                if pd.api.types.is_numeric_dtype(s)
                else "categorical"
            ),
        }

        if pd.api.types.is_numeric_dtype(s):

            x = pd.to_numeric(
                non_missing,
                errors="coerce"
            ).dropna()

            if len(x) > 0:
                record.update({
                    "location_mean": float(x.mean()),
                    "location_median": float(x.median()),
                    "spread_std": float(x.std()),
                    "range_min": float(x.min()),
                    "range_max": float(x.max()),
                    "iqr": float(
                        x.quantile(0.75) -
                        x.quantile(0.25)
                    ),
                    "skewness": float(x.skew()),
                    "kurtosis": float(x.kurtosis()),
                })

                skew = float(x.skew())

                if abs(skew) < 0.5:
                    shape = "approximately_symmetric"
                elif abs(skew) < 1.0:
                    shape = "moderately_skewed"
                else:
                    shape = "highly_skewed"

                record["shape_class"] = shape

        else:

            counts = (
                non_missing
                .astype("object")
                .value_counts(
                    normalize=True
                )
            )

            if len(counts) > 0:
                record.update({
                    "top_category": str(counts.index[0]),
                    "top_category_frequency": float(
                        counts.iloc[0]
                    ),
                    "number_of_categories": int(len(counts)),
                })

        DISTRIBUTION_RECORDS.append(record)

DISTRIBUTION_DF = pd.DataFrame(
    DISTRIBUTION_RECORDS
)

print(
    f"Distribution characterization records: "
    f"{len(DISTRIBUTION_DF)}"
)

print("\nSECTION 10 STATUS: PASS")

SECTION 10 — DISTRIBUTION CHARACTERIZATION


NameError: name 'TRAIN_STATISTICAL_DATASETS' is not defined

In [11]:
# ==============================================================================
# SECTION 11 — PEARSON CORRELATION
# ==============================================================================

print("=" * 100)
print("SECTION 11 — PEARSON CORRELATION")
print("=" * 100)

PEARSON_MATRICES = {}
PEARSON_LONG_RECORDS = []

for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    numeric_columns = FEATURE_TYPE_DF[
        (FEATURE_TYPE_DF["dataset_id"] == dataset_id) &
        (FEATURE_TYPE_DF["semantic_type"] == "numeric")
    ]["feature"].tolist()

    if len(numeric_columns) == 0:
        PEARSON_MATRICES[dataset_id] = pd.DataFrame()
        continue

    corr = (
        df[numeric_columns]
        .corr(method="pearson")
    )

    PEARSON_MATRICES[dataset_id] = corr

    for feature_a in numeric_columns:

        for feature_b in numeric_columns:

            value = corr.loc[
                feature_a,
                feature_b
            ]

            PEARSON_LONG_RECORDS.append({
                "dataset_id": dataset_id,
                "feature_a": feature_a,
                "feature_b": feature_b,
                "pearson_r": float(value)
                if pd.notna(value)
                else np.nan,
            })

PEARSON_DF = pd.DataFrame(
    PEARSON_LONG_RECORDS
)

print(
    f"Pearson matrix pairs: "
    f"{len(PEARSON_DF)}"
)

print("\nSECTION 11 STATUS: PASS")

SECTION 11 — PEARSON CORRELATION


NameError: name 'TRAIN_STATISTICAL_DATASETS' is not defined

In [12]:
# ==============================================================================
# SECTION 12 — SPEARMAN CORRELATION
# ==============================================================================

print("=" * 100)
print("SECTION 12 — SPEARMAN CORRELATION")
print("=" * 100)

SPEARMAN_MATRICES = {}
SPEARMAN_LONG_RECORDS = []

for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    numeric_columns = FEATURE_TYPE_DF[
        (FEATURE_TYPE_DF["dataset_id"] == dataset_id) &
        (FEATURE_TYPE_DF["semantic_type"] == "numeric")
    ]["feature"].tolist()

    if len(numeric_columns) == 0:
        SPEARMAN_MATRICES[dataset_id] = pd.DataFrame()
        continue

    corr = (
        df[numeric_columns]
        .corr(method="spearman")
    )

    SPEARMAN_MATRICES[dataset_id] = corr

    for feature_a in numeric_columns:

        for feature_b in numeric_columns:

            value = corr.loc[
                feature_a,
                feature_b
            ]

            SPEARMAN_LONG_RECORDS.append({
                "dataset_id": dataset_id,
                "feature_a": feature_a,
                "feature_b": feature_b,
                "spearman_rho": float(value)
                if pd.notna(value)
                else np.nan,
            })

SPEARMAN_DF = pd.DataFrame(
    SPEARMAN_LONG_RECORDS
)

print(
    f"Spearman matrix pairs: "
    f"{len(SPEARMAN_DF)}"
)

print("\nSECTION 12 STATUS: PASS")

SECTION 12 — SPEARMAN CORRELATION


NameError: name 'TRAIN_STATISTICAL_DATASETS' is not defined

In [13]:
# ==============================================================================
# SECTION 13 — CATEGORICAL DEPENDENCY ANALYSIS
# ==============================================================================

print("=" * 100)
print("SECTION 13 — CATEGORICAL DEPENDENCY ANALYSIS")
print("=" * 100)

from scipy.stats import chi2_contingency

DEPENDENCY_MAX_ROWS = 25000
DEPENDENCY_MAX_CATEGORIES = 50
DEPENDENCY_RANDOM_SEED = 2025


def prepare_dependency_series(series, max_categories=50):
    """
    Deterministically pool rare categories to prevent enormous contingency
    tables for high-cardinality categorical variables.
    """

    s = series.astype("object").copy()

    s = s.where(
        s.notna(),
        "__MISSING__"
    )

    counts = s.value_counts()

    if len(counts) <= max_categories:
        return s

    keep = set(
        counts.head(max_categories - 1).index
    )

    return s.where(
        s.isin(keep),
        "__OTHER__"
    )


def cramers_v(x, y):
    """
    Bias-corrected Cramér's V.

    Returns a value approximately in [0,1].
    """

    contingency = pd.crosstab(
        x,
        y,
        dropna=False
    )

    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        return np.nan

    observed = contingency.to_numpy(
        dtype=float
    )

    chi2, _, _, _ = chi2_contingency(
        observed,
        correction=False
    )

    n = observed.sum()

    if n <= 1:
        return np.nan

    phi2 = chi2 / n

    r, k = observed.shape

    phi2corr = max(
        0,
        phi2 - (
            (k - 1) *
            (r - 1)
        ) / (n - 1)
    )

    rcorr = (
        r -
        ((r - 1) ** 2) / (n - 1)
    )

    kcorr = (
        k -
        ((k - 1) ** 2) / (n - 1)
    )

    denominator = min(
        kcorr - 1,
        rcorr - 1
    )

    if denominator <= 0:
        return np.nan

    return float(
        np.sqrt(
            phi2corr / denominator
        )
    )


CATEGORICAL_DEPENDENCY_MATRICES = {}
CATEGORICAL_DEPENDENCY_RECORDS = []

for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    categorical_columns = FEATURE_TYPE_DF[
        (FEATURE_TYPE_DF["dataset_id"] == dataset_id) &
        (FEATURE_TYPE_DF["semantic_type"] == "categorical")
    ]["feature"].tolist()

    if len(categorical_columns) < 2:
        CATEGORICAL_DEPENDENCY_MATRICES[
            dataset_id
        ] = pd.DataFrame()
        continue

    if len(df) > DEPENDENCY_MAX_ROWS:

        analysis_df = df.sample(
            n=DEPENDENCY_MAX_ROWS,
            random_state=DEPENDENCY_RANDOM_SEED
        ).copy()

    else:

        analysis_df = df[categorical_columns].copy()

    prepared = {}

    for feature in categorical_columns:
        prepared[feature] = prepare_dependency_series(
            analysis_df[feature],
            max_categories=DEPENDENCY_MAX_CATEGORIES
        )

    matrix = pd.DataFrame(
        np.eye(len(categorical_columns)),
        index=categorical_columns,
        columns=categorical_columns,
        dtype=np.float32
    )

    for i, feature_a in enumerate(categorical_columns):

        for j in range(i + 1, len(categorical_columns)):

            feature_b = categorical_columns[j]

            value = cramers_v(
                prepared[feature_a],
                prepared[feature_b]
            )

            matrix.loc[
                feature_a,
                feature_b
            ] = value

            matrix.loc[
                feature_b,
                feature_a
            ] = value

            CATEGORICAL_DEPENDENCY_RECORDS.append({
                "dataset_id": dataset_id,
                "feature_a": feature_a,
                "feature_b": feature_b,
                "cramers_v": value,
                "analysis_rows": len(analysis_df),
                "category_cap": DEPENDENCY_MAX_CATEGORIES,
            })

    CATEGORICAL_DEPENDENCY_MATRICES[
        dataset_id
    ] = matrix

CATEGORICAL_DEPENDENCY_DF = pd.DataFrame(
    CATEGORICAL_DEPENDENCY_RECORDS
)

print(
    f"Categorical dependency pairs: "
    f"{len(CATEGORICAL_DEPENDENCY_DF)}"
)

print("\nSECTION 13 STATUS: PASS")

SECTION 13 — CATEGORICAL DEPENDENCY ANALYSIS


NameError: name 'TRAIN_STATISTICAL_DATASETS' is not defined

In [14]:
# ==============================================================================
# SECTION 14 — FEATURE STATISTICAL PROFILES
# ==============================================================================

print("=" * 100)
print("SECTION 14 — FEATURE STATISTICAL PROFILES")
print("=" * 100)

FEATURE_STATISTICAL_PROFILES = []

for dataset_id in DATASET_IDS:

    dataset_features = FEATURE_TYPE_DF[
        FEATURE_TYPE_DF["dataset_id"] == dataset_id
    ]

    for _, type_row in dataset_features.iterrows():

        feature = type_row["feature"]
        semantic_type = type_row["semantic_type"]
        is_target = bool(type_row["is_target"])

        missing_row = MISSINGNESS_DF[
            (MISSINGNESS_DF["dataset_id"] == dataset_id) &
            (MISSINGNESS_DF["feature"] == feature)
        ]

        cardinality_row = CARDINALITY_ENTROPY_DF[
            (CARDINALITY_ENTROPY_DF["dataset_id"] == dataset_id) &
            (CARDINALITY_ENTROPY_DF["feature"] == feature)
        ]

        profile = {
            "dataset_id": dataset_id,
            "feature": feature,
            "semantic_type": semantic_type,
            "is_target": is_target,
            "missing_rate": (
                float(
                    missing_row.iloc[0]["missing_rate"]
                )
                if len(missing_row) > 0
                else np.nan
            ),
            "cardinality": (
                int(
                    cardinality_row.iloc[0]["cardinality"]
                )
                if len(cardinality_row) > 0
                else np.nan
            ),
            "normalized_entropy": (
                float(
                    cardinality_row.iloc[0][
                        "normalized_entropy"
                    ]
                )
                if len(cardinality_row) > 0
                else np.nan
            ),
        }

        if semantic_type == "numeric":

            row = NUMERICAL_STATISTICS_DF[
                (NUMERICAL_STATISTICS_DF["dataset_id"] == dataset_id) &
                (NUMERICAL_STATISTICS_DF["feature"] == feature)
            ]

            if len(row) > 0:
                r = row.iloc[0]

                profile.update({
                    "mean": r["mean"],
                    "std": r["std"],
                    "min": r["min"],
                    "q25": r["q25"],
                    "median": r["median"],
                    "q75": r["q75"],
                    "max": r["max"],
                    "skewness": r["skewness"],
                    "kurtosis": r["kurtosis"],
                    "distribution_shape": (
                        DISTRIBUTION_DF[
                            (DISTRIBUTION_DF["dataset_id"] == dataset_id) &
                            (DISTRIBUTION_DF["feature"] == feature)
                        ]["shape_class"].iloc[0]
                        if not DISTRIBUTION_DF[
                            (DISTRIBUTION_DF["dataset_id"] == dataset_id) &
                            (DISTRIBUTION_DF["feature"] == feature)
                        ].empty
                        else None
                    ),
                })

        else:

            row = CATEGORICAL_STATISTICS_DF[
                (CATEGORICAL_STATISTICS_DF["dataset_id"] == dataset_id) &
                (CATEGORICAL_STATISTICS_DF["feature"] == feature)
            ]

            if len(row) > 0:
                r = row.iloc[0]

                profile.update({
                    "top_category": r["top_category"],
                    "top_frequency": r["top_frequency"],
                })

        FEATURE_STATISTICAL_PROFILES.append(
            profile
        )

FEATURE_STATISTICAL_PROFILE_DF = pd.DataFrame(
    FEATURE_STATISTICAL_PROFILES
)

print(
    f"Feature statistical profiles: "
    f"{len(FEATURE_STATISTICAL_PROFILE_DF)}"
)

print("\nSECTION 14 STATUS: PASS")

SECTION 14 — FEATURE STATISTICAL PROFILES


NameError: name 'FEATURE_TYPE_DF' is not defined

In [15]:
# ==============================================================================
# SECTION 15 — DATASET STATISTICAL PROFILES
# ==============================================================================

print("=" * 100)
print("SECTION 15 — DATASET STATISTICAL PROFILES")
print("=" * 100)

DATASET_STATISTICAL_PROFILES = []

for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    feature_profiles = FEATURE_STATISTICAL_PROFILE_DF[
        FEATURE_STATISTICAL_PROFILE_DF["dataset_id"] == dataset_id
    ]

    numeric_profiles = feature_profiles[
        feature_profiles["semantic_type"] == "numeric"
    ]

    categorical_profiles = feature_profiles[
        feature_profiles["semantic_type"] == "categorical"
    ]

    pearson_values = PEARSON_DF[
        (PEARSON_DF["dataset_id"] == dataset_id) &
        (
            PEARSON_DF["feature_a"] !=
            PEARSON_DF["feature_b"]
        )
    ]["pearson_r"].dropna()

    spearman_values = SPEARMAN_DF[
        (SPEARMAN_DF["dataset_id"] == dataset_id) &
        (
            SPEARMAN_DF["feature_a"] !=
            SPEARMAN_DF["feature_b"]
        )
    ]["spearman_rho"].dropna()

    categorical_values = CATEGORICAL_DEPENDENCY_DF[
        CATEGORICAL_DEPENDENCY_DF["dataset_id"] == dataset_id
    ]["cramers_v"].dropna()

    DATASET_STATISTICAL_PROFILES.append({

        "dataset_id": dataset_id,

        "training_rows": len(df),

        "modeling_features": len(
            feature_profiles
        ),

        "numeric_features": int(
            (feature_profiles["semantic_type"] == "numeric").sum()
        ),

        "categorical_features": int(
            (feature_profiles["semantic_type"] == "categorical").sum()
        ),

        "target": TARGET_COLUMNS[dataset_id],

        "mean_feature_missing_rate": float(
            feature_profiles["missing_rate"].mean()
        ),

        "maximum_feature_missing_rate": float(
            feature_profiles["missing_rate"].max()
        ),

        "mean_numeric_skewness": (
            float(numeric_profiles["skewness"].mean())
            if len(numeric_profiles) > 0
            else np.nan
        ),

        "maximum_absolute_pearson": (
            float(pearson_values.abs().max())
            if len(pearson_values) > 0
            else np.nan
        ),

        "maximum_absolute_spearman": (
            float(spearman_values.abs().max())
            if len(spearman_values) > 0
            else np.nan
        ),

        "maximum_categorical_cramers_v": (
            float(categorical_values.max())
            if len(categorical_values) > 0
            else np.nan
        ),
    })

DATASET_STATISTICAL_PROFILE_DF = pd.DataFrame(
    DATASET_STATISTICAL_PROFILES
)

print(
    DATASET_STATISTICAL_PROFILE_DF.to_string(
        index=False
    )
)

print("\nSECTION 15 STATUS: PASS")

SECTION 15 — DATASET STATISTICAL PROFILES


NameError: name 'TRAIN_STATISTICAL_DATASETS' is not defined

In [16]:
# ==============================================================================
# SECTION 16 — BUILD SPP-GAN STATISTICAL REFERENCE
# ==============================================================================

print("=" * 100)
print("SECTION 16 — BUILD SPP-GAN STATISTICAL REFERENCE")
print("=" * 100)

SPP_GAN_STATISTICAL_REFERENCE = {}

REFERENCE_VERSION = "1.0"
REFERENCE_SEED = 2025

for dataset_id in DATASET_IDS:

    feature_profiles = FEATURE_STATISTICAL_PROFILE_DF[
        FEATURE_STATISTICAL_PROFILE_DF["dataset_id"] == dataset_id
    ].copy()

    dataset_profile = DATASET_STATISTICAL_PROFILE_DF[
        DATASET_STATISTICAL_PROFILE_DF["dataset_id"] == dataset_id
    ].iloc[0].to_dict()

    numeric_features = (
        feature_profiles[
            feature_profiles["semantic_type"] == "numeric"
        ]["feature"]
        .tolist()
    )

    categorical_features = (
        feature_profiles[
            feature_profiles["semantic_type"] == "categorical"
        ]["feature"]
        .tolist()
    )

    pearson_matrix = PEARSON_MATRICES[
        dataset_id
    ]

    spearman_matrix = SPEARMAN_MATRICES[
        dataset_id
    ]

    categorical_matrix = (
        CATEGORICAL_DEPENDENCY_MATRICES[
            dataset_id
        ]
    )

    reference = {

        "reference_version": REFERENCE_VERSION,

        "dataset_id": dataset_id,

        "creation_timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat(),

        "random_seed": REFERENCE_SEED,

        "fit_policy": {
            "source_split": "train_only",
            "validation_used_for_reference": False,
            "test_used_for_reference": False,
        },

        "dataset_profile": dataset_profile,

        "feature_schema": {
            "numeric_features": numeric_features,
            "categorical_features": categorical_features,
            "target_column": TARGET_COLUMNS[dataset_id],
            "identifier_columns_excluded": (
                IDENTIFIER_COLUMNS[dataset_id]
            ),
            "provenance_column": PROVENANCE_COLUMN,
        },

        "feature_profiles": (
            feature_profiles
            .replace(
                {np.nan: None}
            )
            .to_dict(
                orient="records"
            )
        ),

        "pearson_correlation": (
            pearson_matrix
            .replace(
                {np.nan: None}
            )
            .to_dict()
            if not pearson_matrix.empty
            else {}
        ),

        "spearman_correlation": (
            spearman_matrix
            .replace(
                {np.nan: None}
            )
            .to_dict()
            if not spearman_matrix.empty
            else {}
        ),

        "categorical_dependency": (
            categorical_matrix
            .replace(
                {np.nan: None}
            )
            .to_dict()
            if not categorical_matrix.empty
            else {}
        ),
    }

    SPP_GAN_STATISTICAL_REFERENCE[
        dataset_id
    ] = reference

print(
    f"Statistical references built: "
    f"{len(SPP_GAN_STATISTICAL_REFERENCE)}"
)

print("\nSECTION 16 STATUS: PASS")

SECTION 16 — BUILD SPP-GAN STATISTICAL REFERENCE


NameError: name 'FEATURE_STATISTICAL_PROFILE_DF' is not defined

In [17]:
# ==============================================================================
# SECTION 17 — GENERATE STATISTICAL GUIDANCE ARTIFACTS
# ==============================================================================

print("=" * 100)
print("SECTION 17 — GENERATE STATISTICAL GUIDANCE ARTIFACTS")
print("=" * 100)

SPP_GAN_STATISTICAL_GUIDANCE = {}

for dataset_id in DATASET_IDS:

    profiles = FEATURE_STATISTICAL_PROFILE_DF[
        FEATURE_STATISTICAL_PROFILE_DF["dataset_id"] == dataset_id
    ].copy()

    numeric_guidance = []
    categorical_guidance = []

    for _, row in profiles.iterrows():

        feature = row["feature"]

        if row["semantic_type"] == "numeric":

            numeric_guidance.append({
                "feature": feature,
                "mean": (
                    float(row["mean"])
                    if pd.notna(row["mean"])
                    else None
                ),
                "std": (
                    float(row["std"])
                    if pd.notna(row["std"])
                    else None
                ),
                "min": (
                    float(row["min"])
                    if pd.notna(row["min"])
                    else None
                ),
                "q25": (
                    float(row["q25"])
                    if pd.notna(row["q25"])
                    else None
                ),
                "median": (
                    float(row["median"])
                    if pd.notna(row["median"])
                    else None
                ),
                "q75": (
                    float(row["q75"])
                    if pd.notna(row["q75"])
                    else None
                ),
                "max": (
                    float(row["max"])
                    if pd.notna(row["max"])
                    else None
                ),
                "skewness": (
                    float(row["skewness"])
                    if pd.notna(row["skewness"])
                    else None
                ),
                "distribution_shape": (
                    row.get("distribution_shape")
                ),
                "missing_rate": float(
                    row["missing_rate"]
                ),
            })

        else:

            categorical_guidance.append({
                "feature": feature,
                "cardinality": int(
                    row["cardinality"]
                )
                if pd.notna(row["cardinality"])
                else None,
                "normalized_entropy": (
                    float(row["normalized_entropy"])
                    if pd.notna(row["normalized_entropy"])
                    else None
                ),
                "top_category": (
                    row.get("top_category")
                ),
                "top_frequency": (
                    float(row["top_frequency"])
                    if pd.notna(row.get("top_frequency", np.nan))
                    else None
                ),
                "missing_rate": float(
                    row["missing_rate"]
                ),
                "is_target": bool(
                    row["is_target"]
                ),
            })

    # --------------------------------------------------------------------------
    # Dependency summaries
    # --------------------------------------------------------------------------

    pearson = PEARSON_DF[
        PEARSON_DF["dataset_id"] == dataset_id
    ].copy()

    pearson = pearson[
        pearson["feature_a"] != pearson["feature_b"]
    ]

    pearson["abs_value"] = (
        pearson["pearson_r"].abs()
    )

    strongest_pearson = (
        pearson
        .sort_values(
            "abs_value",
            ascending=False
        )
        .drop_duplicates(
            subset=[
                "feature_a",
                "feature_b"
            ]
        )
        .head(20)
        .drop(
            columns=["abs_value"]
        )
        .replace({np.nan: None})
        .to_dict(
            orient="records"
        )
    )

    spearman = SPEARMAN_DF[
        SPEARMAN_DF["dataset_id"] == dataset_id
    ].copy()

    spearman = spearman[
        spearman["feature_a"] != spearman["feature_b"]
    ]

    spearman["abs_value"] = (
        spearman["spearman_rho"].abs()
    )

    strongest_spearman = (
        spearman
        .sort_values(
            "abs_value",
            ascending=False
        )
        .drop_duplicates(
            subset=[
                "feature_a",
                "feature_b"
            ]
        )
        .head(20)
        .drop(
            columns=["abs_value"]
        )
        .replace({np.nan: None})
        .to_dict(
            orient="records"
        )
    )

    categorical_dependency = (
        CATEGORICAL_DEPENDENCY_DF[
            CATEGORICAL_DEPENDENCY_DF["dataset_id"] == dataset_id
        ]
        .sort_values(
            "cramers_v",
            ascending=False
        )
        .head(20)
        .replace({np.nan: None})
        .to_dict(
            orient="records"
        )
    )

    SPP_GAN_STATISTICAL_GUIDANCE[
        dataset_id
    ] = {

        "guidance_version": "1.0",

        "dataset_id": dataset_id,

        "source_reference": (
            "SPP_GAN_STATISTICAL_REFERENCE"
        ),

        "source_split": "train_only",

        "numeric_feature_guidance": numeric_guidance,

        "categorical_feature_guidance": (
            categorical_guidance
        ),

        "strongest_numeric_pearson_dependencies": (
            strongest_pearson
        ),

        "strongest_numeric_spearman_dependencies": (
            strongest_spearman
        ),

        "strongest_categorical_dependencies": (
            categorical_dependency
        ),

        "identifier_policy": {
            "excluded": IDENTIFIER_COLUMNS[dataset_id]
        },

        "target_policy": {
            "target": TARGET_COLUMNS[dataset_id],
            "semantic_type": "categorical",
        },

        "provenance_policy": {
            "column": PROVENANCE_COLUMN,
            "excluded_from_model_features": True,
        },
    }

print(
    f"Statistical guidance artifacts prepared: "
    f"{len(SPP_GAN_STATISTICAL_GUIDANCE)}"
)

print("\nSECTION 17 STATUS: PASS")

SECTION 17 — GENERATE STATISTICAL GUIDANCE ARTIFACTS


NameError: name 'FEATURE_STATISTICAL_PROFILE_DF' is not defined

In [18]:
# ==============================================================================
# SECTION 18 — SAVE REPORTS
# ==============================================================================

print("=" * 100)
print("SECTION 18 — SAVE REPORTS")
print("=" * 100)

REPORT_ROOT = NB03_ROOT / "reports"
STATISTICS_ROOT = NB03_ROOT / "statistics"
CORRELATION_ROOT = NB03_ROOT / "correlations"
DEPENDENCY_ROOT = NB03_ROOT / "dependencies"
PROFILE_ROOT = NB03_ROOT / "profiles"
REFERENCE_ROOT = NB03_ROOT / "reference"
GUIDANCE_ROOT = NB03_ROOT / "guidance"

# ------------------------------------------------------------------------------
# Core reports
# ------------------------------------------------------------------------------

DATASET_CHARACTERIZATION_DF.to_csv(
    REPORT_ROOT / "dataset_characterization.csv",
    index=False
)

FEATURE_TYPE_DF.to_csv(
    REPORT_ROOT / "feature_type_characterization.csv",
    index=False
)

FEATURE_TYPE_SUMMARY_DF.to_csv(
    REPORT_ROOT / "feature_type_summary.csv",
    index=False
)

NUMERICAL_STATISTICS_DF.to_csv(
    STATISTICS_ROOT / "numerical_descriptive_statistics.csv",
    index=False
)

CATEGORICAL_STATISTICS_DF.to_csv(
    STATISTICS_ROOT / "categorical_descriptive_statistics.csv",
    index=False
)

MISSINGNESS_DF.to_csv(
    STATISTICS_ROOT / "missingness_characterization.csv",
    index=False
)

MISSINGNESS_SUMMARY_DF.to_csv(
    STATISTICS_ROOT / "missingness_summary.csv",
    index=False
)

CARDINALITY_ENTROPY_DF.to_csv(
    STATISTICS_ROOT / "cardinality_entropy_analysis.csv",
    index=False
)

DISTRIBUTION_DF.to_csv(
    STATISTICS_ROOT / "distribution_characterization.csv",
    index=False
)

FEATURE_STATISTICAL_PROFILE_DF.to_csv(
    PROFILE_ROOT / "feature_statistical_profiles.csv",
    index=False
)

DATASET_STATISTICAL_PROFILE_DF.to_csv(
    PROFILE_ROOT / "dataset_statistical_profiles.csv",
    index=False
)

PEARSON_DF.to_csv(
    CORRELATION_ROOT / "pearson_correlation_long.csv",
    index=False
)

SPEARMAN_DF.to_csv(
    CORRELATION_ROOT / "spearman_correlation_long.csv",
    index=False
)

CATEGORICAL_DEPENDENCY_DF.to_csv(
    DEPENDENCY_ROOT / "categorical_dependency_cramers_v.csv",
    index=False
)

# ------------------------------------------------------------------------------
# Save correlation matrices
# ------------------------------------------------------------------------------

for dataset_id, matrix in PEARSON_MATRICES.items():

    if not matrix.empty:
        matrix.to_csv(
            CORRELATION_ROOT /
            f"{dataset_id}_pearson_matrix.csv"
        )

for dataset_id, matrix in SPEARMAN_MATRICES.items():

    if not matrix.empty:
        matrix.to_csv(
            CORRELATION_ROOT /
            f"{dataset_id}_spearman_matrix.csv"
        )

for dataset_id, matrix in CATEGORICAL_DEPENDENCY_MATRICES.items():

    if not matrix.empty:
        matrix.to_csv(
            DEPENDENCY_ROOT /
            f"{dataset_id}_categorical_cramers_v_matrix.csv"
        )

# ------------------------------------------------------------------------------
# Save statistical reference JSON
# ------------------------------------------------------------------------------

def save_json(data, path):

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            data,
            f,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
            default=str
        )


for dataset_id, reference in (
    SPP_GAN_STATISTICAL_REFERENCE.items()
):

    save_json(
        reference,
        REFERENCE_ROOT /
        f"{dataset_id}_spp_gan_statistical_reference.json"
    )

# ------------------------------------------------------------------------------
# Save guidance JSON
# ------------------------------------------------------------------------------

for dataset_id, guidance in (
    SPP_GAN_STATISTICAL_GUIDANCE.items()
):

    save_json(
        guidance,
        GUIDANCE_ROOT /
        f"{dataset_id}_spp_gan_statistical_guidance.json"
    )

print("\nSaved:")
print(f"  Reports       : {REPORT_ROOT}")
print(f"  Statistics    : {STATISTICS_ROOT}")
print(f"  Correlations  : {CORRELATION_ROOT}")
print(f"  Dependencies  : {DEPENDENCY_ROOT}")
print(f"  Profiles      : {PROFILE_ROOT}")
print(f"  Reference     : {REFERENCE_ROOT}")
print(f"  Guidance      : {GUIDANCE_ROOT}")

print("\nSECTION 18 STATUS: PASS")

SECTION 18 — SAVE REPORTS


NameError: name 'DATASET_CHARACTERIZATION_DF' is not defined

In [19]:
# ==============================================================================
# SECTION 19 — VALIDATE STATISTICAL ARTIFACTS
# ==============================================================================

print("=" * 100)
print("SECTION 19 — VALIDATE STATISTICAL ARTIFACTS")
print("=" * 100)

STATISTICAL_ARTIFACT_VALIDATION = []

# ------------------------------------------------------------------------------
# Expected artifacts
# ------------------------------------------------------------------------------

expected_files = [

    REPORT_ROOT / "dataset_characterization.csv",
    REPORT_ROOT / "feature_type_characterization.csv",
    REPORT_ROOT / "feature_type_summary.csv",

    STATISTICS_ROOT / "numerical_descriptive_statistics.csv",
    STATISTICS_ROOT / "categorical_descriptive_statistics.csv",
    STATISTICS_ROOT / "missingness_characterization.csv",
    STATISTICS_ROOT / "missingness_summary.csv",
    STATISTICS_ROOT / "cardinality_entropy_analysis.csv",
    STATISTICS_ROOT / "distribution_characterization.csv",

    PROFILE_ROOT / "feature_statistical_profiles.csv",
    PROFILE_ROOT / "dataset_statistical_profiles.csv",

    CORRELATION_ROOT / "pearson_correlation_long.csv",
    CORRELATION_ROOT / "spearman_correlation_long.csv",

    DEPENDENCY_ROOT / "categorical_dependency_cramers_v.csv",
]

for path in expected_files:

    exists = path.exists()

    STATISTICAL_ARTIFACT_VALIDATION.append({
        "artifact": str(path),
        "exists": exists,
        "size_bytes": (
            path.stat().st_size
            if exists
            else 0
        ),
    })

# ------------------------------------------------------------------------------
# Validate JSON references/guidance
# ------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    reference_path = (
        REFERENCE_ROOT /
        f"{dataset_id}_spp_gan_statistical_reference.json"
    )

    guidance_path = (
        GUIDANCE_ROOT /
        f"{dataset_id}_spp_gan_statistical_guidance.json"
    )

    for artifact_name, path in [
        ("statistical_reference", reference_path),
        ("statistical_guidance", guidance_path),
    ]:

        STATISTICAL_ARTIFACT_VALIDATION.append({
            "artifact": str(path),
            "exists": path.exists(),
            "size_bytes": (
                path.stat().st_size
                if path.exists()
                else 0
            ),
        })

# ------------------------------------------------------------------------------
# Structural validation
# ------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    reference = SPP_GAN_STATISTICAL_REFERENCE[
        dataset_id
    ]

    guidance = SPP_GAN_STATISTICAL_GUIDANCE[
        dataset_id
    ]

    required_reference_keys = {
        "reference_version",
        "dataset_id",
        "fit_policy",
        "dataset_profile",
        "feature_schema",
        "feature_profiles",
        "pearson_correlation",
        "spearman_correlation",
        "categorical_dependency",
    }

    required_guidance_keys = {
        "guidance_version",
        "dataset_id",
        "source_reference",
        "source_split",
        "numeric_feature_guidance",
        "categorical_feature_guidance",
        "strongest_numeric_pearson_dependencies",
        "strongest_numeric_spearman_dependencies",
        "strongest_categorical_dependencies",
    }

    if not required_reference_keys.issubset(
        reference.keys()
    ):
        raise RuntimeError(
            f"{dataset_id}: statistical reference schema invalid."
        )

    if not required_guidance_keys.issubset(
        guidance.keys()
    ):
        raise RuntimeError(
            f"{dataset_id}: statistical guidance schema invalid."
        )

    if reference["fit_policy"]["source_split"] != "train_only":
        raise RuntimeError(
            f"{dataset_id}: statistical reference is not train-only."
        )

    if guidance["source_split"] != "train_only":
        raise RuntimeError(
            f"{dataset_id}: guidance is not train-only."
        )

# ------------------------------------------------------------------------------
# Validate numerical correlation matrices
# ------------------------------------------------------------------------------

for dataset_id, matrix in PEARSON_MATRICES.items():

    if not matrix.empty:

        if list(matrix.index) != list(matrix.columns):
            raise RuntimeError(
                f"{dataset_id}: Pearson matrix is not square."
            )

        if not np.allclose(
            matrix.to_numpy(dtype=float),
            matrix.to_numpy(dtype=float).T,
            equal_nan=True
        ):
            raise RuntimeError(
                f"{dataset_id}: Pearson matrix is not symmetric."
            )

for dataset_id, matrix in SPEARMAN_MATRICES.items():

    if not matrix.empty:

        if list(matrix.index) != list(matrix.columns):
            raise RuntimeError(
                f"{dataset_id}: Spearman matrix is not square."
            )

        if not np.allclose(
            matrix.to_numpy(dtype=float),
            matrix.to_numpy(dtype=float).T,
            equal_nan=True
        ):
            raise RuntimeError(
                f"{dataset_id}: Spearman matrix is not symmetric."
            )

# ------------------------------------------------------------------------------
# Validate Cramér's V
# ------------------------------------------------------------------------------

if not CATEGORICAL_DEPENDENCY_DF.empty:

    invalid_cramers = (
        CATEGORICAL_DEPENDENCY_DF[
            CATEGORICAL_DEPENDENCY_DF["cramers_v"].notna()
        ]["cramers_v"]
        .apply(lambda x: not (0.0 <= x <= 1.0))
        .any()
    )

    if invalid_cramers:
        raise RuntimeError(
            "Invalid Cramér's V values detected."
        )

# ------------------------------------------------------------------------------
# Final artifact existence gate
# ------------------------------------------------------------------------------

STATISTICAL_ARTIFACT_VALIDATION_DF = pd.DataFrame(
    STATISTICAL_ARTIFACT_VALIDATION
)

missing_artifacts = (
    STATISTICAL_ARTIFACT_VALIDATION_DF[
        ~STATISTICAL_ARTIFACT_VALIDATION_DF["exists"]
    ]
)

if len(missing_artifacts) > 0:

    print(
        missing_artifacts.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "One or more statistical artifacts are missing."
    )

STATISTICAL_ARTIFACT_VALIDATION_DF.to_csv(
    REPORT_ROOT / "statistical_artifact_validation.csv",
    index=False
)

print(
    f"\nValidated artifacts: "
    f"{len(STATISTICAL_ARTIFACT_VALIDATION_DF)}"
)

print("\nSECTION 19 STATUS: PASS")

SECTION 19 — VALIDATE STATISTICAL ARTIFACTS


KeyError: 'adult_income'

In [20]:
# ==============================================================================
# SECTION 20 — FINAL VERIFICATION
# ==============================================================================

print("=" * 100)
print("SECTION 20 — FINAL VERIFICATION")
print("=" * 100)

FINAL_VERIFICATION_RECORDS = []

def add_final_check(
    check_name,
    status,
    details=""
):
    FINAL_VERIFICATION_RECORDS.append({
        "check": check_name,
        "status": "PASS" if status else "FAIL",
        "details": details,
    })


# ------------------------------------------------------------------------------
# 1. Dataset registry
# ------------------------------------------------------------------------------

add_final_check(
    "dataset_registry",
    set(DATASET_IDS) == {
        "adult_income",
        "bank_marketing",
        "diabetes_130us",
    },
    f"Datasets: {DATASET_IDS}"
)


# ------------------------------------------------------------------------------
# 2. Training datasets
# ------------------------------------------------------------------------------

add_final_check(
    "training_datasets_loaded",
    all(
        dataset_id in TRAIN_STATISTICAL_DATASETS
        for dataset_id in DATASET_IDS
    )
)


# ------------------------------------------------------------------------------
# 3. Input schema validation
# ------------------------------------------------------------------------------

add_final_check(
    "input_schema_validation",
    (
        "status" in INPUT_SCHEMA_SUMMARY_DF.columns
        and
        (INPUT_SCHEMA_SUMMARY_DF["status"] == "PASS").all()
    )
)


# ------------------------------------------------------------------------------
# 4. Dataset characterization
# ------------------------------------------------------------------------------

add_final_check(
    "dataset_characterization",
    len(DATASET_CHARACTERIZATION_DF) == len(DATASET_IDS)
)


# ------------------------------------------------------------------------------
# 5. Feature characterization
# ------------------------------------------------------------------------------

expected_feature_counts = {
    "adult_income": 15,
    "bank_marketing": 17,
    "diabetes_130us": 48,
}

feature_count_check = True

for dataset_id, expected_count in (
    expected_feature_counts.items()
):

    actual = int(
        (
            FEATURE_TYPE_DF["dataset_id"] == dataset_id
        ).sum()
    )

    if actual != expected_count:
        feature_count_check = False

add_final_check(
    "feature_type_characterization",
    feature_count_check,
    f"Expected feature counts: {expected_feature_counts}"
)


# ------------------------------------------------------------------------------
# 6. Statistical profiles
# ------------------------------------------------------------------------------

add_final_check(
    "feature_statistical_profiles",
    all(
        dataset_id in
        set(
            FEATURE_STATISTICAL_PROFILE_DF["dataset_id"]
        )
        for dataset_id in DATASET_IDS
    )
)


add_final_check(
    "dataset_statistical_profiles",
    len(DATASET_STATISTICAL_PROFILE_DF) == len(DATASET_IDS)
)


# ------------------------------------------------------------------------------
# 7. Correlation artifacts
# ------------------------------------------------------------------------------

add_final_check(
    "pearson_correlation",
    len(PEARSON_DF) >= 0
)

add_final_check(
    "spearman_correlation",
    len(SPEARMAN_DF) >= 0
)


# ------------------------------------------------------------------------------
# 8. Categorical dependency
# ------------------------------------------------------------------------------

add_final_check(
    "categorical_dependency",
    len(CATEGORICAL_DEPENDENCY_DF) >= 0
)


# ------------------------------------------------------------------------------
# 9. Statistical reference
# ------------------------------------------------------------------------------

reference_check = True

for dataset_id in DATASET_IDS:

    if dataset_id not in SPP_GAN_STATISTICAL_REFERENCE:
        reference_check = False
        break

    if (
        SPP_GAN_STATISTICAL_REFERENCE[dataset_id]
        ["fit_policy"]
        ["source_split"]
        != "train_only"
    ):
        reference_check = False
        break

add_final_check(
    "spp_gan_statistical_reference",
    reference_check
)


# ------------------------------------------------------------------------------
# 10. Statistical guidance
# ------------------------------------------------------------------------------

guidance_check = (
    set(SPP_GAN_STATISTICAL_GUIDANCE.keys())
    == set(DATASET_IDS)
)

add_final_check(
    "spp_gan_statistical_guidance",
    guidance_check
)


# ------------------------------------------------------------------------------
# 11. Leakage policy
# ------------------------------------------------------------------------------

leakage_check = True

for dataset_id in DATASET_IDS:

    reference = SPP_GAN_STATISTICAL_REFERENCE[
        dataset_id
    ]

    if reference["fit_policy"][
        "validation_used_for_reference"
    ]:
        leakage_check = False

    if reference["fit_policy"][
        "test_used_for_reference"
    ]:
        leakage_check = False

add_final_check(
    "train_only_reference_policy",
    leakage_check
)


# ------------------------------------------------------------------------------
# 12. Identifier policy
# ------------------------------------------------------------------------------

identifier_policy_check = True

for dataset_id in DATASET_IDS:

    profile_features = set(
        FEATURE_STATISTICAL_PROFILE_DF[
            FEATURE_STATISTICAL_PROFILE_DF[
                "dataset_id"
            ] == dataset_id
        ]["feature"]
    )

    for identifier in IDENTIFIER_COLUMNS[dataset_id]:

        if identifier in profile_features:
            identifier_policy_check = False

add_final_check(
    "identifier_exclusion_policy",
    identifier_policy_check
)


# ------------------------------------------------------------------------------
# 13. Provenance exclusion
# ------------------------------------------------------------------------------

provenance_check = (
    PROVENANCE_COLUMN
    not in
    set(FEATURE_STATISTICAL_PROFILE_DF["feature"])
)

add_final_check(
    "provenance_exclusion",
    provenance_check
)


# ------------------------------------------------------------------------------
# 14. Artifact completeness
# ------------------------------------------------------------------------------

artifact_completeness = (
    len(
        STATISTICAL_ARTIFACT_VALIDATION_DF[
            ~STATISTICAL_ARTIFACT_VALIDATION_DF["exists"]
        ]
    )
    == 0
)

add_final_check(
    "artifact_completeness",
    artifact_completeness
)


# ------------------------------------------------------------------------------
# Build final verification dataframe
# ------------------------------------------------------------------------------

FINAL_VERIFICATION_DF = pd.DataFrame(
    FINAL_VERIFICATION_RECORDS
)

OVERALL_NOTEBOOK_03_PASS = (
    FINAL_VERIFICATION_DF["status"] == "PASS"
).all()

FINAL_VERIFICATION_DF.to_csv(
    REPORT_ROOT /
    "notebook_03_final_verification.csv",
    index=False
)


# ------------------------------------------------------------------------------
# Completion metadata
# ------------------------------------------------------------------------------

completion_metadata = {

    "notebook": "03",

    "title": (
        "Statistical & Data Characterization"
    ),

    "status": (
        "COMPLETE"
        if OVERALL_NOTEBOOK_03_PASS
        else "FAILED"
    ),

    "creation_timestamp_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),

    "datasets": DATASET_IDS,

    "statistical_reference_policy": {
        "source_split": "train_only",
        "validation_used": False,
        "test_used": False,
    },

    "dataset_characterization_rows": int(
        len(DATASET_CHARACTERIZATION_DF)
    ),

    "feature_profile_rows": int(
        len(FEATURE_STATISTICAL_PROFILE_DF)
    ),

    "pearson_pairs": int(
        len(PEARSON_DF)
    ),

    "spearman_pairs": int(
        len(SPEARMAN_DF)
    ),

    "categorical_dependency_pairs": int(
        len(CATEGORICAL_DEPENDENCY_DF)
    ),

    "statistical_reference_datasets": int(
        len(SPP_GAN_STATISTICAL_REFERENCE)
    ),

    "statistical_guidance_datasets": int(
        len(SPP_GAN_STATISTICAL_GUIDANCE)
    ),

    "overall_pass": bool(
        OVERALL_NOTEBOOK_03_PASS
    ),
}

completion_path = (
    NB03_ROOT /
    "schemas" /
    "notebook_03_completion_metadata.json"
)

save_json(
    completion_metadata,
    completion_path
)


# ------------------------------------------------------------------------------
# Print final gate
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 03 FINAL VERIFICATION")
print("=" * 100)

print(
    FINAL_VERIFICATION_DF.to_string(
        index=False
    )
)

print("\n" + "-" * 100)

if OVERALL_NOTEBOOK_03_PASS:

    print("NOTEBOOK 03 STATUS: COMPLETE")
    print("NOTEBOOK 03 COMPLETION GATE: PASS")

else:

    failed = FINAL_VERIFICATION_DF[
        FINAL_VERIFICATION_DF["status"] != "PASS"
    ]

    print(
        failed.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "NOTEBOOK 03 COMPLETION GATE FAILED."
    )

print("\nCompletion metadata saved to:")
print(completion_path)

print("\nNotebook 03 is COMPLETE and ready for downstream SPP-GAN notebooks.")

SECTION 20 — FINAL VERIFICATION


NameError: name 'TRAIN_STATISTICAL_DATASETS' is not defined

In [21]:
# ==============================================================================
# SECTION 21 — COMPLETION SUMMARY
# ==============================================================================
#
# PURPOSE
# -------
# Final human-readable and machine-readable completion summary for Notebook 03.
#
# IMPORTANT
# ---------
# This section:
#   • does NOT recompute statistics
#   • does NOT modify the statistical reference
#   • does NOT use validation/test data
#   • does NOT train any model
#   • does NOT generate synthetic data
#   • does NOT alter Notebook 02 artifacts
#
# It only summarizes and verifies the already completed Notebook 03 pipeline.
#
# ==============================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
import numpy as np

print("=" * 100)
print("SECTION 21 — COMPLETION SUMMARY")
print("=" * 100)


# ==============================================================================
# 1. VERIFY REQUIRED NOTEBOOK 03 OBJECTS
# ==============================================================================

REQUIRED_NOTEBOOK_03_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_STATISTICAL_DATASETS",
    "DATASET_CHARACTERIZATION_DF",
    "FEATURE_TYPE_DF",
    "NUMERICAL_STATISTICS_DF",
    "CATEGORICAL_STATISTICS_DF",
    "MISSINGNESS_DF",
    "CARDINALITY_ENTROPY_DF",
    "DISTRIBUTION_DF",
    "PEARSON_DF",
    "SPEARMAN_DF",
    "CATEGORICAL_DEPENDENCY_DF",
    "FEATURE_STATISTICAL_PROFILE_DF",
    "DATASET_STATISTICAL_PROFILE_DF",
    "SPP_GAN_STATISTICAL_REFERENCE",
    "SPP_GAN_STATISTICAL_GUIDANCE",
    "STATISTICAL_ARTIFACT_VALIDATION_DF",
    "FINAL_VERIFICATION_DF",
    "OVERALL_NOTEBOOK_03_PASS",
]

missing_objects = [
    name
    for name in REQUIRED_NOTEBOOK_03_OBJECTS
    if name not in globals()
]

if missing_objects:

    print("\nMissing required objects:")
    for item in missing_objects:
        print(f"  ✗ {item}")

    raise RuntimeError(
        "Notebook 03 completion summary cannot proceed because "
        "required objects are missing."
    )

print("\nRequired Notebook 03 objects : PASS")


# ==============================================================================
# 2. VERIFY FINAL VERIFICATION GATE
# ==============================================================================

if not bool(OVERALL_NOTEBOOK_03_PASS):

    print(
        "\nFINAL VERIFICATION GATE : FAIL"
    )

    failed_checks = FINAL_VERIFICATION_DF[
        FINAL_VERIFICATION_DF["status"] != "PASS"
    ]

    print(
        failed_checks.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Notebook 03 final verification gate failed."
    )

print("Final verification gate       : PASS")


# ==============================================================================
# 3. DATASET COMPLETENESS
# ==============================================================================

dataset_completeness_records = []

for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    feature_profiles = FEATURE_STATISTICAL_PROFILE_DF[
        FEATURE_STATISTICAL_PROFILE_DF["dataset_id"] == dataset_id
    ]

    reference_exists = (
        dataset_id in SPP_GAN_STATISTICAL_REFERENCE
    )

    guidance_exists = (
        dataset_id in SPP_GAN_STATISTICAL_GUIDANCE
    )

    dataset_profile_exists = (
        dataset_id in set(
            DATASET_STATISTICAL_PROFILE_DF[
                "dataset_id"
            ]
        )
    )

    dataset_completeness_records.append({
        "dataset_id": dataset_id,
        "training_rows": len(df),
        "statistical_features": len(feature_profiles),
        "reference_present": reference_exists,
        "guidance_present": guidance_exists,
        "dataset_profile_present": dataset_profile_exists,
        "status": (
            "PASS"
            if (
                reference_exists
                and guidance_exists
                and dataset_profile_exists
            )
            else "FAIL"
        ),
    })

DATASET_COMPLETENESS_DF = pd.DataFrame(
    dataset_completeness_records
)

if (
    DATASET_COMPLETENESS_DF["status"] != "PASS"
).any():

    print(
        DATASET_COMPLETENESS_DF.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Dataset completeness verification failed."
    )

print("Dataset completeness            : PASS")


# ==============================================================================
# 4. STATISTICAL COVERAGE SUMMARY
# ==============================================================================

STATISTICAL_COVERAGE_SUMMARY = {

    "datasets": len(DATASET_IDS),

    "dataset_characterization_rows": int(
        len(DATASET_CHARACTERIZATION_DF)
    ),

    "feature_type_rows": int(
        len(FEATURE_TYPE_DF)
    ),

    "numerical_statistics_rows": int(
        len(NUMERICAL_STATISTICS_DF)
    ),

    "categorical_statistics_rows": int(
        len(CATEGORICAL_STATISTICS_DF)
    ),

    "missingness_rows": int(
        len(MISSINGNESS_DF)
    ),

    "cardinality_entropy_rows": int(
        len(CARDINALITY_ENTROPY_DF)
    ),

    "distribution_rows": int(
        len(DISTRIBUTION_DF)
    ),

    "pearson_rows": int(
        len(PEARSON_DF)
    ),

    "spearman_rows": int(
        len(SPEARMAN_DF)
    ),

    "categorical_dependency_rows": int(
        len(CATEGORICAL_DEPENDENCY_DF)
    ),

    "feature_profile_rows": int(
        len(FEATURE_STATISTICAL_PROFILE_DF)
    ),

    "dataset_profile_rows": int(
        len(DATASET_STATISTICAL_PROFILE_DF)
    ),

    "statistical_reference_datasets": int(
        len(SPP_GAN_STATISTICAL_REFERENCE)
    ),

    "statistical_guidance_datasets": int(
        len(SPP_GAN_STATISTICAL_GUIDANCE)
    ),
}


# ==============================================================================
# 5. PRINT DATASET SUMMARY
# ==============================================================================

print("\n" + "-" * 100)
print("DATASET SUMMARY")
print("-" * 100)

for dataset_id in DATASET_IDS:

    row = DATASET_COMPLETENESS_DF[
        DATASET_COMPLETENESS_DF["dataset_id"] == dataset_id
    ].iloc[0]

    dataset_profile = DATASET_CHARACTERIZATION_DF[
        DATASET_CHARACTERIZATION_DF["dataset_id"] == dataset_id
    ].iloc[0]

    feature_profile_count = int(
        row["statistical_features"]
    )

    numeric_count = int(
        dataset_profile["numeric_columns"]
    )

    categorical_count = int(
        dataset_profile["categorical_columns"]
    )

    target = TARGET_COLUMNS[dataset_id]

    identifiers = IDENTIFIER_COLUMNS[dataset_id]

    print(f"\n{dataset_id}:")
    print(
        f"  Training rows        : "
        f"{int(row['training_rows']):,}"
    )
    print(
        f"  Modeling features    : "
        f"{feature_profile_count}"
    )
    print(
        f"  Numeric features     : "
        f"{numeric_count}"
    )
    print(
        f"  Categorical features : "
        f"{categorical_count}"
    )
    print(
        f"  Target               : "
        f"{target}"
    )
    print(
        f"  Identifiers excluded : "
        f"{', '.join(identifiers) if identifiers else 'None'}"
    )
    print(
        f"  Statistical reference: "
        f"{'PASS' if row['reference_present'] else 'FAIL'}"
    )
    print(
        f"  Statistical guidance : "
        f"{'PASS' if row['guidance_present'] else 'FAIL'}"
    )
    print(
        f"  Dataset status       : "
        f"{row['status']}"
    )


# ==============================================================================
# 6. PRINT STATISTICAL COVERAGE
# ==============================================================================

print("\n" + "-" * 100)
print("STATISTICAL COVERAGE")
print("-" * 100)

print(
    f"Datasets                         : "
    f"{STATISTICAL_COVERAGE_SUMMARY['datasets']}"
)

print(
    f"Feature statistical profiles    : "
    f"{STATISTICAL_COVERAGE_SUMMARY['feature_profile_rows']}"
)

print(
    f"Numerical statistics records    : "
    f"{STATISTICAL_COVERAGE_SUMMARY['numerical_statistics_rows']}"
)

print(
    f"Categorical statistics records  : "
    f"{STATISTICAL_COVERAGE_SUMMARY['categorical_statistics_rows']}"
)

print(
    f"Missingness records             : "
    f"{STATISTICAL_COVERAGE_SUMMARY['missingness_rows']}"
)

print(
    f"Cardinality/entropy records     : "
    f"{STATISTICAL_COVERAGE_SUMMARY['cardinality_entropy_rows']}"
)

print(
    f"Distribution records            : "
    f"{STATISTICAL_COVERAGE_SUMMARY['distribution_rows']}"
)

print(
    f"Pearson correlation records     : "
    f"{STATISTICAL_COVERAGE_SUMMARY['pearson_rows']}"
)

print(
    f"Spearman correlation records    : "
    f"{STATISTICAL_COVERAGE_SUMMARY['spearman_rows']}"
)

print(
    f"Categorical dependency records  : "
    f"{STATISTICAL_COVERAGE_SUMMARY['categorical_dependency_rows']}"
)


# ==============================================================================
# 7. VERIFY TRAIN-ONLY STATISTICAL POLICY
# ==============================================================================

train_only_policy_pass = True

for dataset_id in DATASET_IDS:

    reference = SPP_GAN_STATISTICAL_REFERENCE[
        dataset_id
    ]

    guidance = SPP_GAN_STATISTICAL_GUIDANCE[
        dataset_id
    ]

    reference_policy = reference.get(
        "fit_policy",
        {}
    )

    if reference_policy.get(
        "source_split"
    ) != "train_only":
        train_only_policy_pass = False

    if reference_policy.get(
        "validation_used_for_reference"
    ) is not False:
        train_only_policy_pass = False

    if reference_policy.get(
        "test_used_for_reference"
    ) is not False:
        train_only_policy_pass = False

    if guidance.get(
        "source_split"
    ) != "train_only":
        train_only_policy_pass = False


if not train_only_policy_pass:

    raise RuntimeError(
        "Train-only statistical reference policy failed."
    )

print("\nTrain-only statistical policy   : PASS")


# ==============================================================================
# 8. VERIFY IDENTIFIER / PROVENANCE POLICY
# ==============================================================================

identifier_provenance_pass = True

for dataset_id in DATASET_IDS:

    feature_names = set(
        FEATURE_STATISTICAL_PROFILE_DF[
            FEATURE_STATISTICAL_PROFILE_DF["dataset_id"] == dataset_id
        ]["feature"]
    )

    for identifier in IDENTIFIER_COLUMNS[dataset_id]:

        if identifier in feature_names:
            identifier_provenance_pass = False

    if PROVENANCE_COLUMN in feature_names:
        identifier_provenance_pass = False


if not identifier_provenance_pass:

    raise RuntimeError(
        "Identifier/provenance exclusion policy failed."
    )

print("Identifier/provenance policy    : PASS")


# ==============================================================================
# 9. VERIFY TARGET POLICY
# ==============================================================================

target_policy_pass = True

for dataset_id in DATASET_IDS:

    target = TARGET_COLUMNS[dataset_id]

    target_rows = FEATURE_STATISTICAL_PROFILE_DF[
        (FEATURE_STATISTICAL_PROFILE_DF["dataset_id"] == dataset_id) &
        (FEATURE_STATISTICAL_PROFILE_DF["feature"] == target)
    ]

    if len(target_rows) != 1:
        target_policy_pass = False
        continue

    target_row = target_rows.iloc[0]

    if target_row["semantic_type"] != "categorical":
        target_policy_pass = False

    if not bool(target_row["is_target"]):
        target_policy_pass = False


if not target_policy_pass:

    raise RuntimeError(
        "Target characterization policy failed."
    )

print("Target characterization policy  : PASS")


# ==============================================================================
# 10. VERIFY REFERENCE / GUIDANCE CONSISTENCY
# ==============================================================================

reference_guidance_consistency = True

for dataset_id in DATASET_IDS:

    reference = SPP_GAN_STATISTICAL_REFERENCE[
        dataset_id
    ]

    guidance = SPP_GAN_STATISTICAL_GUIDANCE[
        dataset_id
    ]

    if reference["dataset_id"] != dataset_id:
        reference_guidance_consistency = False

    if guidance["dataset_id"] != dataset_id:
        reference_guidance_consistency = False

    reference_features = set(
        reference["feature_schema"][
            "numeric_features"
        ]
    ).union(
        reference["feature_schema"][
            "categorical_features"
        ]
    )

    profile_features = set(
        FEATURE_STATISTICAL_PROFILE_DF[
            FEATURE_STATISTICAL_PROFILE_DF[
                "dataset_id"
            ] == dataset_id
        ]["feature"]
    )

    if reference_features != profile_features:
        reference_guidance_consistency = False


if not reference_guidance_consistency:

    raise RuntimeError(
        "Statistical reference/guidance consistency failed."
    )

print("Reference/guidance consistency   : PASS")


# ==============================================================================
# 11. VERIFY ALL PERSISTED ARTIFACTS
# ==============================================================================

artifact_validation_pass = (
    len(
        STATISTICAL_ARTIFACT_VALIDATION_DF[
            ~STATISTICAL_ARTIFACT_VALIDATION_DF["exists"]
        ]
    ) == 0
)

if not artifact_validation_pass:

    missing_artifacts = (
        STATISTICAL_ARTIFACT_VALIDATION_DF[
            ~STATISTICAL_ARTIFACT_VALIDATION_DF["exists"]
        ]
    )

    print(
        missing_artifacts.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Statistical artifact completeness failed."
    )

print("Statistical artifact completeness: PASS")


# ==============================================================================
# 12. SAVE DATASET COMPLETENESS REPORT
# ==============================================================================

DATASET_COMPLETENESS_PATH = (
    NB03_ROOT
    / "reports"
    / "dataset_completeness.csv"
)

DATASET_COMPLETENESS_DF.to_csv(
    DATASET_COMPLETENESS_PATH,
    index=False
)


# ==============================================================================
# 13. SAVE STATISTICAL COVERAGE REPORT
# ==============================================================================

STATISTICAL_COVERAGE_DF = pd.DataFrame(
    [
        {
            "metric": key,
            "value": value,
        }
        for key, value in
        STATISTICAL_COVERAGE_SUMMARY.items()
    ]
)

STATISTICAL_COVERAGE_PATH = (
    NB03_ROOT
    / "reports"
    / "statistical_coverage_summary.csv"
)

STATISTICAL_COVERAGE_DF.to_csv(
    STATISTICAL_COVERAGE_PATH,
    index=False
)


# ==============================================================================
# 14. FINAL COMPLETION SUMMARY
# ==============================================================================

NOTEBOOK_03_COMPLETION_SUMMARY = {

    "notebook_number": "03",

    "notebook_title": (
        "Statistical & Data Characterization"
    ),

    "status": "COMPLETE",

    "completion_gate": "PASS",

    "completion_timestamp_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),

    "project_root": str(
        PROJECT_ROOT
    ),

    "notebook_02_dependency": (
        "Notebook 02 frozen artifacts"
    ),

    "datasets": DATASET_IDS,

    "statistical_source_policy": {
        "source_split": "train_only",
        "validation_used": False,
        "test_used": False,
    },

    "feature_policy": {
        "identifiers_excluded": True,
        "provenance_excluded": True,
        "target_retained": True,
        "target_semantic_type": "categorical",
    },

    "statistical_methods": [
        "dataset_characterization",
        "feature_type_characterization",
        "numerical_descriptive_statistics",
        "categorical_descriptive_statistics",
        "missingness_characterization",
        "cardinality_analysis",
        "entropy_analysis",
        "distribution_characterization",
        "pearson_correlation",
        "spearman_correlation",
        "cramers_v_categorical_dependency",
    ],

    "statistical_reference": {
        "datasets": len(
            SPP_GAN_STATISTICAL_REFERENCE
        ),
        "status": "COMPLETE",
    },

    "statistical_guidance": {
        "datasets": len(
            SPP_GAN_STATISTICAL_GUIDANCE
        ),
        "status": "COMPLETE",
    },

    "coverage": STATISTICAL_COVERAGE_SUMMARY,

    "validation": {
        "final_verification": "PASS",
        "dataset_completeness": "PASS",
        "train_only_policy": "PASS",
        "identifier_provenance_policy": "PASS",
        "target_policy": "PASS",
        "reference_guidance_consistency": "PASS",
        "artifact_completeness": "PASS",
    },

    "ready_for_downstream": True,

    "downstream_use": [
        "SPP-GAN architecture",
        "statistical conditioning/reference",
        "training-time statistical guidance",
        "synthetic-data evaluation",
        "comparative baseline analysis",
    ],
}


# ==============================================================================
# 15. SAVE FINAL COMPLETION SUMMARY
# ==============================================================================

COMPLETION_SUMMARY_PATH = (
    NB03_ROOT
    / "schemas"
    / "notebook_03_completion_summary.json"
)

with open(
    COMPLETION_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        NOTEBOOK_03_COMPLETION_SUMMARY,
        f,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
        default=str,
    )


# ==============================================================================
# 16. SAVE FINAL STATUS REPORT
# ==============================================================================

FINAL_STATUS_RECORD = {

    "notebook": "03",

    "status": "COMPLETE",

    "completion_gate": "PASS",

    "datasets": len(DATASET_IDS),

    "feature_profiles": len(
        FEATURE_STATISTICAL_PROFILE_DF
    ),

    "dataset_profiles": len(
        DATASET_STATISTICAL_PROFILE_DF
    ),

    "pearson_records": len(
        PEARSON_DF
    ),

    "spearman_records": len(
        SPEARMAN_DF
    ),

    "categorical_dependency_records": len(
        CATEGORICAL_DEPENDENCY_DF
    ),

    "statistical_reference_datasets": len(
        SPP_GAN_STATISTICAL_REFERENCE
    ),

    "statistical_guidance_datasets": len(
        SPP_GAN_STATISTICAL_GUIDANCE
    ),

    "train_only": True,

    "identifiers_excluded": True,

    "provenance_excluded": True,

    "target_retained": True,

    "artifact_completeness": True,

    "ready_for_notebook_04": True,

    "timestamp_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
}

FINAL_STATUS_PATH = (
    NB03_ROOT
    / "reports"
    / "notebook_03_final_status.csv"
)

pd.DataFrame(
    [FINAL_STATUS_RECORD]
).to_csv(
    FINAL_STATUS_PATH,
    index=False
)


# ==============================================================================
# 17. FINAL CONSOLE SUMMARY
# ==============================================================================

print("\n")
print("=" * 100)
print("NOTEBOOK 03 — COMPLETION SUMMARY")
print("=" * 100)

print(
    "\nNotebook                              : "
    "03 — Statistical & Data Characterization"
)

print(
    "Datasets                               : "
    f"{len(DATASET_IDS)}"
)

print(
    "Feature statistical profiles           : "
    f"{len(FEATURE_STATISTICAL_PROFILE_DF)}"
)

print(
    "Dataset statistical profiles           : "
    f"{len(DATASET_STATISTICAL_PROFILE_DF)}"
)

print(
    "Pearson correlation records            : "
    f"{len(PEARSON_DF)}"
)

print(
    "Spearman correlation records           : "
    f"{len(SPEARMAN_DF)}"
)

print(
    "Categorical dependency records         : "
    f"{len(CATEGORICAL_DEPENDENCY_DF)}"
)

print(
    "SPP-GAN statistical references         : "
    f"{len(SPP_GAN_STATISTICAL_REFERENCE)}"
)

print(
    "SPP-GAN statistical guidance artifacts : "
    f"{len(SPP_GAN_STATISTICAL_GUIDANCE)}"
)

print(
    "Statistical source                     : "
    "TRAIN ONLY"
)

print(
    "Validation used for reference          : "
    "NO"
)

print(
    "Test used for reference                : "
    "NO"
)

print(
    "Identifiers included as features       : "
    "NO"
)

print(
    "Provenance included as feature         : "
    "NO"
)

print(
    "Target retained in characterization    : "
    "YES"
)

print("\n" + "-" * 100)

print(
    "Required objects                       : PASS"
)

print(
    "Final verification gate                : PASS"
)

print(
    "Dataset completeness                   : PASS"
)

print(
    "Train-only statistical policy          : PASS"
)

print(
    "Identifier/provenance policy           : PASS"
)

print(
    "Target characterization policy         : PASS"
)

print(
    "Reference/guidance consistency         : PASS"
)

print(
    "Statistical artifact completeness      : PASS"
)

print(
    "Downstream readiness                   : PASS"
)

print("\n" + "=" * 100)
print("NOTEBOOK 03 STATUS: COMPLETE")
print("=" * 100)

print(
    "\nNOTEBOOK 03 COMPLETION GATE PASSED"
)

print(
    "\nCompletion summary saved to:"
)

print(
    COMPLETION_SUMMARY_PATH
)

print(
    "\nFinal status report saved to:"
)

print(
    FINAL_STATUS_PATH
)

print(
    "\nNotebook 03 is COMPLETE and ready for downstream SPP-GAN notebooks."
)

print("=" * 100)

SECTION 21 — COMPLETION SUMMARY

Missing required objects:
  ✗ TRAIN_STATISTICAL_DATASETS
  ✗ DATASET_CHARACTERIZATION_DF
  ✗ FEATURE_TYPE_DF
  ✗ NUMERICAL_STATISTICS_DF
  ✗ CATEGORICAL_STATISTICS_DF
  ✗ MISSINGNESS_DF
  ✗ CARDINALITY_ENTROPY_DF
  ✗ DISTRIBUTION_DF
  ✗ PEARSON_DF
  ✗ SPEARMAN_DF
  ✗ CATEGORICAL_DEPENDENCY_DF
  ✗ FEATURE_STATISTICAL_PROFILE_DF
  ✗ DATASET_STATISTICAL_PROFILE_DF
  ✗ STATISTICAL_ARTIFACT_VALIDATION_DF
  ✗ FINAL_VERIFICATION_DF
  ✗ OVERALL_NOTEBOOK_03_PASS


RuntimeError: Notebook 03 completion summary cannot proceed because required objects are missing.